# 1. Environment & Imports

โหลด autoreload สำหรับให้ Notebook รับโค้ดใหม่จาก `src/` อัตโนมัติเมื่อแก้ไข และ import `TextCleaner` ซึ่งเป็น Pipeline หลักสำหรับทำความสะอาดข้อมูล

In [27]:
%load_ext autoreload
%autoreload 2

import sys
import os

import pandas as pd

sys.path.append(os.path.abspath('../'))
from src.data_cleaning import TextCleaner


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 2. Data Loading & Inspection

โหลดข้อมูลตัวอย่าง (`sample.csv`, ไม่มี header) และตรวจสอบโครงสร้าง/สัดส่วน Missing Values ของข้อมูลดิบก่อนเข้าสู่ขั้นตอนทำความสะอาด

In [28]:
raw_data = pd.read_csv(
    '../data/raw/amazon_review_polarity_csv/sample.csv',
    header=None,
    index_col=False,
)
raw_data.head()


,0,1,2
0,2,Stuning even for the non-gamer,This sound track was beautiful! It paints the ...
1,2,The best soundtrack ever to anything.,I'm reading a lot of reviews saying that this ...
2,2,Amazing!,This soundtrack is my favorite music of all ti...
3,2,Excellent Soundtrack,I truly like this soundtrack and I enjoy video...
4,2,"Remember, Pull Your Jaw Off The Floor After He...","If you've played the game, you know how divine..."


In [29]:
raw_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   0       50000 non-null  int64
 1   1       49995 non-null  str  
 2   2       50000 non-null  str  
dtypes: int64(1), str(2)
memory usage: 1.1 MB


# 3. Data Cleaning Pipeline

รัน Pipeline หลักจาก `TextCleaner` (`src/data_cleaning.py`): rename/map คอลัมน์, แปลง dtype, ล้างข้อความ (HTML/URL, whitespace, contractions), รวม `title` + `text` เป็นคอลัมน์ `review`, และกรอง duplicate/ข้อความสั้นเกินไปออก

In [30]:
cleaner = TextCleaner()
cleaned_data = cleaner.processing(raw_data)


Removed 0 duplicate reviews
Removed 0 empty/short reviews (< 3 chars)
Rows: 50000 -> 50000
Missing values per column:
sentiment    0
title        0
text         0
review       0
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   sentiment  50000 non-null  category
 1   title      50000 non-null  str     
 2   text       50000 non-null  str     
 3   review     50000 non-null  str     
dtypes: category(1), str(3)
memory usage: 49.1 MB


# 4. Data Quality Check & Results

ตรวจผลลัพธ์หลังทำความสะอาด — Validation Report (จำนวนแถวก่อน/หลัง, duplicate ที่ถูกลบ, Missing Values, Memory Usage) ถูก print ไว้แล้วในขั้นตอนก่อนหน้าโดย `processing()`; cell นี้ใช้ตรวจตัวอย่างผลลัพธ์แบบ Interactive เพิ่มเติม

In [31]:
cleaned_data.head()


,sentiment,title,text,review
0,positive,stuning even for the non-gamer,this sound track was beautiful! it paints the ...,stuning even for the non-gamer this sound trac...
1,positive,the best soundtrack ever to anything.,i am reading a lot of reviews saying that this...,the best soundtrack ever to anything. i am rea...
2,positive,amazing!,this soundtrack is my favorite music of all ti...,amazing! this soundtrack is my favorite music ...
3,positive,excellent soundtrack,i truly like this soundtrack and i enjoy video...,excellent soundtrack i truly like this soundtr...
4,positive,"remember, pull your jaw off the floor after he...","if you have played the game, you know how divi...","remember, pull your jaw off the floor after he..."


ตรวจว่ายังมีคำที่มี apostrophe contraction (เช่น `n't`) หลงเหลืออยู่ในคอลัมน์ `review` หลังผ่าน Cleaning Pipeline หรือไม่ — ใช้เป็น Diagnostic เพื่อดูช่องว่าง ของการ Normalize คำย่อ/คำสะกดผิดที่ยังต้องปรับปรุงต่อ

In [32]:
cleaned_data[cleaned_data['review'].str.contains("n't")]

,sentiment,title,text,review
222,positive,rare find,"a good book for audi owners, and fans in gener...","rare find a good book for audi owners, and fan..."
239,positive,one of her best in the night world series!,out of all l.j.smiths works this is one of my ...,one of her best in the night world series! out...
503,positive,textbook,should have been a nationwide elementary schoo...,textbook should have been a nationwide element...
1035,negative,just horrible,i really cannot understand why all the reviewe...,just horrible i really cannot understand why a...
2242,negative,huh?,i just do not get it? i thought this book was ...,huh? i just do not get it? i thought this book...
...,...,...,...,...
48354,negative,made me sick,it had a good taste but i got sick from drinki...,made me sick it had a good taste but i got sic...
48554,negative,the abyss - a real loser!,you know and i know this movie is so boring it...,the abyss - a real loser! you know and i know ...
48985,positive,this vacuum is great,"wanted to say it ""sucks"" but not in the slang ...","this vacuum is great wanted to say it ""sucks"" ..."
49648,positive,pipi,what can i say that hasen't already been said....,pipi what can i say that hasen't already been ...


สังเกตว่าจำนวนแถวข้างบนไม่ได้ลดลงจากการเพิ่ม ftfy เพราะ regex นี้ค้นหาเฉพาะ apostrophe แบบตรง (`'`) แต่ ftfy แก้ปัญหาคนละจุด: คำย่อที่พิมพ์ด้วย apostrophe โค้ง/มน (`’`, เช่น `don’t`, `I’m`) ซึ่งไม่เคยถูกตรวจพบทั้งใน regex นี้และใน `contractions.fix()` เลยตั้งแต่แรก ตัวอย่างด้านล่างแสดงให้เห็นความแตกต่างก่อน/หลังเพิ่ม `fix_encoding()`

In [34]:
raw_curly_example = "So I\u2019m short about 19 0r so words for your review."
print('RAW           :', raw_curly_example)
print('CLEANED (now) :', cleaner.clean_text(pd.Series([raw_curly_example]))[0])


RAW           : So I’m short about 19 0r so words for your review.
CLEANED (now) : so i am short about 19 0r so words for your review.
